In [1]:
%load_ext dotenv
%dotenv

In [18]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.tools import tool
from langchain_core.tools import create_retriever_tool

from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langsmith import Client

from platform import python_version

In [9]:
wikipedia_tool = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper())

In [5]:
vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",
                     embedding_function= HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k':3, 'lambda_mul':0.7})

retriever_tool = create_retriever_tool(retriever = retriever,
                                       name = "Intro to Data and Data Science Course Lectures",
                                       description = '''For any questions regarding the Introduction
                                       to Data and Data Science course, you must use this tool.''')

d:\projects\Langchain module tools and agents\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1624.13it/s]
C:\Users\muham\AppData\Local\Temp\ipykernel_20184\263501917.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",


In [6]:
@tool("Another Name")
def get_python_version() -> str:
    """Useful for questions regarding the version of Python currently used."""
    return python_version()

In [11]:
tools = [wikipedia_tool, retriever_tool, get_python_version]

In [13]:
chat = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [20]:
client = Client()
chat_prompt_template = client.pull_prompt(
    "hwchase17/openai-tools-agent",
    dangerously_pull_public_prompt=True
)

In [21]:
chat_prompt_template

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [22]:
chat_prompt_template.pretty_print()

================================ System Message ================================

You are a helpful assistant

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}

============================= Messages Placeholder =============================

{agent_scratchpad}
